In [ ]:
%pip install -e ../../../
%pip install plotly
%pip install nbformat

In [ ]:
import numpy as np
from numpy.random import default_rng
RNG = default_rng(12809)
import pandas as pd

from water_futures_battle.core import Settings
from water_futures_battle.jurisdictions import build_state
from water_futures_battle.jurisdictions.entities import Province
from water_futures_battle.sources import build_sources
settings_desc = {
    "start_year": 2000,
    "end_year": 2024,
    "national_budget": 0,
    "lifeline_volume": 80,
    "seed": 128
}
nl_desc = {
    'name': 'Netherlands',
    'id': 'NL0000',
    'jurisdictions-static_properties': 'jurisdictions-static_properties.xlsx',
    'municipalities-dynamic_properties': 'municipalities-dynamic_properties.xlsx'
}
netherlands = build_state(
    nl_desc,
    data_path='../../jurisdictions/',
    settings=Settings.from_config(settings_desc)
    )

# Upload the demands generated in the `jurisdicitons-view_demands-historical` notebook
pvs_max_daily_dem_df = pd.read_csv(
    '../../jurisdictions/provinces-max_daily_demands-CMD.csv',
    index_col='year'
)
juris_max_hourly_dem_df = pd.read_csv(
    '../../jurisdictions/jurisdictions-max_hourly_demands-CMH.csv',
    index_col='year',
    parse_dates=True
)
juris_max_hourly_dem_df.index = juris_max_hourly_dem_df.index.year

sources_desc = {
    'sources-static_properties': 'sources-static_properties.xlsx',
    'groundwater-dynamic_properties': 'groundwater-dynamic_properties.xlsx',
    'surface_water-dynamic_properties': 'surface_water-dynamic_properties.xlsx',
    'desalination-dynamic_properties': 'desalination-dynamic_properties.xlsx',
    'groundwater-construction_size-max_permit_surplus': 0.3,
    'surface_water-construction_size-min': 0,
    'surface_water-construction_size-max': 0,
    'desalination-construction_size-min': 0,
    'desalination-construction_size-max': 0,
}
all_sources, sources_settings = build_sources(
    sources_desc,
    a_state=netherlands,
    data_path='../../sources/',
    settings=Settings.from_config(settings_desc)
    )
sources_lookup = pd.read_excel(
    '../../sources/sources-static_properties.xlsx',
    sheet_name=list(all_sources.keys()),
    index_col='source_id'
)

provinces_peak2source_factor = { }
for pv in netherlands.provinces:
    peak_demands__CMH = juris_max_hourly_dem_df[pv.cbs_id]
    
    province_sources = all_sources.source_by_province(pv)
    capacities__CMH = []
    for year in peak_demands__CMH.index:
        active_sources = [s for s in province_sources if s.is_active(when=year)]

        capacities__CMH.append(sum([s.nominal_capacity/24 for s in active_sources]))

    ratios = peak_demands__CMH.values / np.array(capacities__CMH)
    
    provinces_peak2source_factor[pv] = max(ratios.max()*1.15, 1.5)


pump_options = pd.read_excel(
    '../../pumps/pump_options-static_properties.xlsx',
    index_col='option_id',
    sheet_name='options'
)
pump_curves = pd.read_excel(
    '../../pumps/pump_options-static_properties.xlsx',
    sheet_name=None
)
pump_options['flow_rate-maximum'] = pump_options.index.map(
    lambda pcid: pump_curves[pcid]['flow_rate'].max()
)

COLUMNS_PSTAT = [
    'pumping_station_id',
    'assigned_source',
    'pumps-option_ids',
    'pumps-installation_dates',
    'pumps-end_dates'
]

def pstat_id_gen(x: int) -> str:
    return f"PS{x:04d}"

# The logic to size the pumps
TARGET_N_PUMPS = 7
def pump_option_selection(options_df: pd.DataFrame, design_q: float) -> tuple[str, int]:
    # temp pump option, nominal flow rate

    n_pumps: dict[str, int] = {
        pump_id: int(np.ceil(design_q/options_df.loc[pump_id, 'flow_rate-maximum']))
        for pump_id in options_df.index
    }

    option_id = next(
        (pump_id  for pump_id, n in n_pumps.items() if n <= TARGET_N_PUMPS),      
        list(n_pumps.keys())[-1]
    )

    return (option_id, n_pumps[option_id])

In [ ]:
# check if closing year of pump is earlier than closing year of source
def append_to_end_years_if_valid(end_years, year_to_add, source):
    if not pd.isnull(source.closure_date):
        # source closed earlier, so pump closure year is the same as source closure year
        if year_to_add > source.closure_date.year:
            end_years.append(f"{source.closure_date.year}-01-01")
            return end_years
    if year_to_add >= 2025:
        # not valid, so pump is still active
        end_years.append(f"")
        return end_years
    # else just add year to end_years
    end_years.append(f"{year_to_add}-01-01")
    return end_years

In [ ]:
# Let's allocate and name all pumping stations. No pumps associated for now
pstat_df = pd.DataFrame(columns=COLUMNS_PSTAT)
pump_lifetime_max = 30

for source in all_sources.ordered_entities:

    pstat_id = pstat_id_gen(len(pstat_df) +1)
    pstat_source_id = source.bwf_id

    if not np.isnan(source.nominal_capacity):
        source_cap_cmh = source.nominal_capacity/24
        target_flow = source_cap_cmh * provinces_peak2source_factor[source.province]

        selected_pumpopt, quantity = pump_option_selection(pump_options, target_flow)

        if source.activation_date.year >= 1975:
            # at most one iteration of new pumps possible

            # all pumps installed in activation year of source
            installation_years = [f"{source.activation_date.year}-01-01"
                                  for _ in range(quantity)]
            end_years = []

            if source.activation_date.year < 2000:
                # a previous generation of pumps exists
                # sampling of lifetime of previous pump generation
                ages = RNG.integers(
                    pump_options.loc[selected_pumpopt, 'lifetime-min'].astype(int),
                    pump_lifetime_max +1,
                    size=(quantity,)
                )
                ages = list(sorted(ages))
                for age in ages:
                    # add currently active pump (= second generation of pump for this source)
                    new_pump_install_year = source.activation_date.year+age
                    if (
                        new_pump_install_year < 2025 and
                        (pd.isna(source.closure_date) or new_pump_install_year < source.closure_date.year)
                    ):
                        # otherwise the old pump is still active, so we do not have to install a new
                        installation_years.append(f"{new_pump_install_year}-01-01")
                    end_years = append_to_end_years_if_valid(
                        end_years,
                        new_pump_install_year,
                        source
                    )

        elif source.activation_date.year < 1975:
            # multiple generations of pumps exist

            # Sampling of ages of currently installed pumps (between 2000 and 2024)
            ages = RNG.integers(1, 25, size=(quantity,))
            # Sampling of lifetimes of previous installed pumps
            old_pumps_lifetimes = RNG.integers(
                pump_options.loc[selected_pumpopt, 'lifetime-min'].astype(int),
                pump_lifetime_max +1,
                size=(quantity,)
            )
            # Compute installation years of previous pumps
            overall_ages = [ages[pump]+old_pumps_lifetimes[pump]
                            for pump in range(quantity)]
            ages = ages[np.argsort(overall_ages)[::-1]]
            overall_ages = list(sorted(overall_ages, reverse=True))
            # add previous generation of pumps
            installation_years = [f"{2025-overall_age}-01-01"
                                  for overall_age in overall_ages]
            end_years = []
            for age in ages:
                # add end years of currently installed pumps
                end_years = append_to_end_years_if_valid(
                    end_years,
                    2025-age,
                    source
                )
            # add new pumps sorted by age
            ages = list(sorted(ages, reverse=True))
            for age in ages:
                # add sorted installation years of currently installed pumps
                new_pump_install_year = 2025-age
                if  (
                    new_pump_install_year < 2025 and 
                    (pd.isna(source.closure_date) or new_pump_install_year < source.closure_date.year)
                ):
                    installation_years.append(f"{new_pump_install_year}-01-01")

        # delete unnecessary ";" from end_dates
        while end_years != [] and end_years[-1] == "":
            end_years.pop()

        if pd.notna(source.closure_date):
            n_closing_pumps = len(installation_years) - len(end_years)
            end_years = end_years + [
                source.closure_date.strftime("%Y-%m-%d")
            ]*n_closing_pumps

        # create pump_options_id
        installed_pumps = [selected_pumpopt]*len(installation_years)


    else: # source has not been opened yet, therefore empty string info
        installation_years = []
        installed_pumps = []
        end_years = []

    pstat_df.loc[len(pstat_df)] = [
        pstat_id,
        pstat_source_id,
        ';'.join(installed_pumps),
        ';'.join(installation_years),
        ';'.join(end_years)
    ]

In [ ]:
# Save out
from water_futures_battle.core.base_model import StaticProperties
pstat_sps = StaticProperties('pumping_stations-static_properties')

pstat_sps['entities'] = pstat_df

pstat_sps.dump(path='..')

In [ ]:
pstat_df['n_pumps'] = pstat_df[
    ['pumps-installation_dates', 'pumps-end_dates']
].apply(
    lambda row: len(row.iloc[0].split(";"))-len(row.iloc[1].split(";")),
    axis=1
)
pstat_df['unique_ptype'] = pstat_df['pumps-option_ids'].apply(
    lambda x: x.split(';')[0]
)

import plotly.express as px

# Sort by pump type and number of pumps for better visualization
pstat_df_sorted = pstat_df.sort_values(['unique_ptype', 'n_pumps'])

# Create a row index for x-axis
pstat_df_sorted = pstat_df_sorted.reset_index(drop=True)

# Create bar chart with each row as a bar, colored by pump type
fig = px.bar(
    pstat_df_sorted,
    x=pstat_df_sorted.index,
    y='n_pumps',
    color='unique_ptype',
    title='Distribution of Number of Pumps by Type',
    labels={
        'x': 'Pump Station',
        'n_pumps': 'Number of Pumps',
        'unique_ptype': 'Pump Type'
    },
    hover_data=['unique_ptype', 'n_pumps']
)

# Update layout
fig.update_layout(
    xaxis_title='Pump Station (sorted by type)',
    yaxis_title='Number of Pumps',
    showlegend=True
)

fig.show()